# Nettoyage de `goods_receipts.csv`

Ce notebook diagnostique et nettoie les réceptions de marchandises. Les corrections sont limitées aux anomalies certaines ; les valeurs manquantes non déductibles sont conservées.

## 0. Bibliothèques et chemins

In [ ]:
from pathlib import Path
import re
import pandas as pd

CURRENT_DIR = Path.cwd()
DATA_DIR = CURRENT_DIR / 'data' / 'logistics' if (CURRENT_DIR / 'data' / 'logistics').exists() else CURRENT_DIR
SOURCE_FILE = DATA_DIR / 'goods_receipts.csv'
OUTPUT_FILE = DATA_DIR / 'goods_receipts_cleaned.csv'

## 1. Chargement et aperçu général

In [ ]:
df_raw = pd.read_csv(SOURCE_FILE, dtype=str)
df = df_raw.copy()
print(f'Dimensions : {df.shape[0]} lignes et {df.shape[1]} colonnes')
display(df.head())
display(df.dtypes.rename('type initial'))

## 2. Valeurs manquantes

On mesure les absences avant de décider si elles peuvent être corrigées sans inventer d'information.

In [ ]:
missing = pd.DataFrame({
    'nombre': df.isna().sum(),
    'pourcentage': (df.isna().mean() * 100).round(2)
})
display(missing.loc[missing['nombre'] > 0])
display(df.loc[df['inspection_status'].isna()])

Les quatre inspections manquantes concernent des réceptions `posted`. Elles ne sont pas imputées : une réception publiée peut avoir une inspection `accepted` ou `conditional`, donc aucune valeur unique ne peut être déduite.

## 3. Doublons

In [ ]:
duplicate_mask = df.duplicated(keep=False)
print('Lignes appartenant à un groupe de doublons :', duplicate_mask.sum())
print('Copies supplémentaires à retirer :', df.duplicated().sum())
display(df.loc[duplicate_mask].sort_values('receipt_number'))

Les doublons sont strictement identiques. Garder la première occurrence ne supprime donc aucune information distincte.

In [ ]:
df = df.drop_duplicates(keep='first').copy()
print('Nombre de lignes après déduplication :', len(df))

## 4. Contrôle et nettoyage des formats

### 4.1 Identifiants

In [ ]:
identifier_patterns = {
    'receipt_number': r'^RCV-\d{4}-\d{5}$',
    'purchase_order_number': r'^PO-\d{4}-\d{5}$',
    'receiving_facility_code': r'^FCL-\d{4}$',
    'supplier_partner_code': r'^BP-\d{5}$',
    'batch_reference': r'^B-\d{4}-\d{5}$',
    'storage_location_code': r'^BIN-\d{2}$',
}
for column, pattern in identifier_patterns.items():
    invalid = ~df[column].str.match(pattern, na=False)
    print(f'{column}: {invalid.sum()} format(s) invalide(s)')
print('Identifiants receipt_number dupliqués :', df['receipt_number'].duplicated().sum())

### 4.2 Dates

Cinq dates utilisent `/` au lieu de `-`. Les formats année/mois/jour et jour/mois/année sont traités explicitement pour éviter toute ambiguïté.

In [ ]:
standard_date_pattern = r'^\d{4}-\d{2}-\d{2}$'
non_standard_dates = ~df['receipt_date'].str.match(standard_date_pattern, na=False)
display(df.loc[non_standard_dates, ['receipt_number', 'receipt_date']])

def parse_receipt_date(value):
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', value):
        return pd.to_datetime(value, format='%Y-%m-%d')
    if re.fullmatch(r'\d{4}/\d{2}/\d{2}', value):
        return pd.to_datetime(value, format='%Y/%m/%d')
    if re.fullmatch(r'\d{2}/\d{2}/\d{4}', value):
        return pd.to_datetime(value, format='%d/%m/%Y')
    return pd.NaT

df['receipt_date'] = df['receipt_date'].map(parse_receipt_date)
print('Dates non convertibles restantes :', df['receipt_date'].isna().sum())

### 4.3 Quantité reçue

Les anomalies sont des virgules décimales ou la répétition de l'unité dans la valeur. L'unité intégrée est contrôlée avant d'être retirée.

In [ ]:
quantity_as_number = pd.to_numeric(df['received_quantity'], errors='coerce')
bad_quantity_mask = quantity_as_number.isna() & df['received_quantity'].notna()
display(df.loc[bad_quantity_mask, ['receipt_number', 'received_quantity', 'received_unit']])

embedded_unit = df['received_quantity'].str.extract(r'[0-9]\s+([A-Za-z_]+)\s*$', expand=False)
unit_conflict = embedded_unit.notna() & embedded_unit.str.casefold().ne(df['received_unit'].str.casefold())
print('Unités intégrées en conflit avec received_unit :', unit_conflict.sum())

In [ ]:
clean_quantity = (df['received_quantity']
                  .str.strip()
                  .str.replace(',', '.', regex=False)
                  .str.replace(r'\s+[A-Za-z_]+\s*$', '', regex=True))
df['received_quantity'] = pd.to_numeric(clean_quantity, errors='raise')
df['purchase_order_line'] = pd.to_numeric(df['purchase_order_line'], errors='raise').astype('Int64')
display(df['received_quantity'].describe())

### 4.4 Variables catégorielles

In [ ]:
categorical_columns = ['received_unit', 'inspection_status', 'receipt_status']
for column in categorical_columns:
    print(f'\n{column}')
    display(df[column].value_counts(dropna=False))

for column in categorical_columns:
    df[column] = df[column].str.strip().str.casefold()

print('Modalités après harmonisation :')
for column in categorical_columns:
    print(column, sorted(df[column].dropna().unique()))

## 5. Cohérence avec les tables de référence

Les commandes, fournisseurs et établissements doivent exister dans les autres tables logistiques. On vérifie aussi que le fournisseur, l'établissement et l'unité correspondent à la commande d'achat.

In [ ]:
purchase_orders = pd.read_csv(DATA_DIR / 'purchase_order_lines.csv', dtype=str).drop_duplicates()
facilities = pd.read_csv(DATA_DIR / 'erp_facilities.csv', dtype=str)
partners = pd.read_csv(DATA_DIR / 'erp_business_partners.csv', dtype=str)

po_keys = ['purchase_order_number', 'purchase_order_line']
purchase_orders['purchase_order_line'] = pd.to_numeric(purchase_orders['purchase_order_line'], errors='raise').astype('Int64')
purchase_orders = purchase_orders.drop_duplicates(po_keys)
comparison = df.merge(purchase_orders, on=po_keys, how='left', suffixes=('_receipt', '_order'), validate='many_to_one', indicator=True)

print('Commandes introuvables :', comparison['_merge'].eq('left_only').sum())
print('Fournisseurs incohérents :', comparison['supplier_partner_code_receipt'].ne(comparison['supplier_partner_code_order']).sum())
print('Établissements incohérents :', comparison['receiving_facility_code'].ne(comparison['ordering_facility_code']).sum())
print('Unités incohérentes :', comparison['received_unit'].ne(comparison['ordered_unit'].str.casefold()).sum())
print('Codes établissement inconnus :', (~df['receiving_facility_code'].isin(facilities['facility_code'])).sum())
print('Codes fournisseur inconnus :', (~df['supplier_partner_code'].isin(partners['partner_code'])).sum())

## 6. Contrôles finaux

Les valeurs extrêmes positives sont conservées : elles restent dans la plage des commandes associées et ne constituent pas, à elles seules, des erreurs.

In [ ]:
assert df.shape == (610, 12)
assert df['receipt_number'].is_unique
assert not df.duplicated().any()
assert df['receipt_date'].notna().all()
assert df['received_quantity'].gt(0).all()
assert df['inspection_status'].isna().sum() == 4
assert set(df['inspection_status'].dropna()) <= {'accepted', 'conditional', 'held'}
assert set(df['receipt_status']) <= {'posted', 'quality_hold'}
assert (df.loc[df['inspection_status'].eq('held'), 'receipt_status'] == 'quality_hold').all()
assert comparison['_merge'].eq('both').all()
assert comparison['supplier_partner_code_receipt'].eq(comparison['supplier_partner_code_order']).all()
assert comparison['receiving_facility_code'].eq(comparison['ordering_facility_code']).all()
assert comparison['received_unit'].eq(comparison['ordered_unit'].str.casefold()).all()
print('Tous les contrôles finaux sont passés.')
display(df.info())

## 7. Export du fichier nettoyé

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d', na_rep='')
print(f'Fichier créé : {OUTPUT_FILE}')
print(f'Dimensions finales : {df.shape[0]} lignes et {df.shape[1]} colonnes')